In [1]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS


# 1. Charger le dataset

def charger_dataset():
    df = pd.read_csv("amazon.csv")
    print(df.head())
    print(df.info())
    return df



# Stopwords

stop_words = set(ENGLISH_STOP_WORDS)

def supprimer_stopwords(texte):
    mots = texte.split()
    mots_filtrés = [mot for mot in mots if mot not in stop_words]
    return " ".join(mots_filtrés)



# 2. Nettoyer les données

def nettoyer_donnees(df):
    df["review"] = df["review"].str.lower()
    df["review"] = df["review"].str.replace(r"[^a-zA-Z\s]", "", regex=True)
    df["review"] = df["review"].apply(supprimer_stopwords)
    return df



# 3. Séparer les données

def separer_donnees(df):
    X_train, X_test, y_train, y_test = train_test_split(
        df["review"], df["resultat"], test_size=0.2, random_state=42
    )
    return X_train, X_test, y_train, y_test



# 4. Vectorisation TF-IDF

def vectoriser_texte(X_train, X_test):
    vectoriseur = TfidfVectorizer()

    X_train_tfidf = vectoriseur.fit_transform(X_train)
    X_test_tfidf = vectoriseur.transform(X_test)

    return X_train_tfidf, X_test_tfidf, vectoriseur


# 5. Entraîner le modèle

def entrainer_modele(X_train_tfidf, y_train):
    modele = LogisticRegression()
    modele.fit(X_train_tfidf, y_train)
    return modele



# 6. Évaluer le modèle

def evaluer_modele(modele, X_test_tfidf, y_test):
    y_pred = modele.predict(X_test_tfidf)

    print("\nRapport de classification:\n")
    print(classification_report(y_test, y_pred))

    print("\nMatrice de confusion:\n")
    print(confusion_matrix(y_test, y_pred))


# MAIN

data = charger_dataset()
data = nettoyer_donnees(data)

X_train, X_test, y_train, y_test = separer_donnees(data)

X_train_tfidf, X_test_tfidf, vectoriseur = vectoriser_texte(X_train, X_test)

modele = entrainer_modele(X_train_tfidf, y_train)

evaluer_modele(modele, X_test_tfidf, y_test)

                                              review  sentiment resultat
0  This is  the best apps acording to a bunch of ...          1  positif
1  This is a pretty good version of the game for ...          1  positif
2  this is a really . there are a bunch of levels...          1  positif
3  This is a silly game and can be frustrating, b...          1  positif
4  This is a terrific game on any pad. Hrs of fun...          1  positif
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19996 entries, 0 to 19995
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     19996 non-null  object
 1   sentiment  19996 non-null  int64 
 2   resultat   19996 non-null  object
dtypes: int64(1), object(2)
memory usage: 468.8+ KB
None

Rapport de classification:

              precision    recall  f1-score   support

     négatif       0.86      0.62      0.72       978
     positif       0.89      0.97      0.93      3022

    accur

In [2]:
import re
import gradio as gr


# 7. Prédire le sentiment

def predire_sentiment(avis, modele, vectoriseur):
    # Nettoyage
    avis_nettoye = avis.lower()
    avis_nettoye = re.sub(r"[^a-zA-Z\s]", "", avis_nettoye)
    avis_nettoye = supprimer_stopwords(avis_nettoye)

    # Vectorisation
    avis_tfidf = vectoriseur.transform([avis_nettoye])

    # Prédiction
    prediction = modele.predict(avis_tfidf)[0]

    return f"Résultat : {prediction}"




# 8. Interface Gradio
def lancer_application():
    # Charger les données
    df = charger_dataset()
    df = nettoyer_donnees(df)
    # Split
    X_train, X_test, y_train, y_test = separer_donnees(df)
    # TF-IDF
    X_train_tfidf, X_test_tfidf, vectoriseur = vectoriser_texte(X_train, X_test)
    # Modèle
    modele = entrainer_modele(X_train_tfidf, y_train)
    # Évaluation
    evaluer_modele(modele, X_test_tfidf, y_test)


    # Fonction pour interface
    def interface_gradio(avis):
        return predire_sentiment(avis, modele, vectoriseur)

    # Interface Gradio
    app = gr.Interface(
        fn=interface_gradio,
        inputs=gr.Textbox(lines=2, placeholder="Entrez un avis ici..."),
        outputs="text",
        title="Analyse des Sentiments",
        description="Taper Votre Sentiments"
    )
    app.launch()




# Lancement

if __name__ == "__main__":
    lancer_application()

                                              review  sentiment resultat
0  This is  the best apps acording to a bunch of ...          1  positif
1  This is a pretty good version of the game for ...          1  positif
2  this is a really . there are a bunch of levels...          1  positif
3  This is a silly game and can be frustrating, b...          1  positif
4  This is a terrific game on any pad. Hrs of fun...          1  positif
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19996 entries, 0 to 19995
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     19996 non-null  object
 1   sentiment  19996 non-null  int64 
 2   resultat   19996 non-null  object
dtypes: int64(1), object(2)
memory usage: 468.8+ KB
None

Rapport de classification:

              precision    recall  f1-score   support

     négatif       0.86      0.62      0.72       978
     positif       0.89      0.97      0.93      3022

    accur